In [3]:
!pip install numpy opencv-python

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/16.3 MB 10.3 MB/s eta 0:00:02
   ----------- ---------------------------- 4.7/16.3 MB 11.0 MB/s eta 0:00:02
   ------------------ --------------------- 7.3/16.3 MB 11.3 MB/s eta 0:00:01
   ------------------------ --------------- 10.0/16.3 MB 11.5 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.3 MB 11.6 MB/s eta 0:00:01
   ------------------------------------ --- 14.9/16.3 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 11.0 MB/s eta 0:00:00


In [1]:
import numpy as np
import os

def load_npy_files(directory):
    npy_files = [f for f in os.listdir(directory) if f.endswith('metric_depth.npy')]
    data_list = []
    for file in npy_files:
        data = np.load(os.path.join(directory, file))
        data_list.append(data)
    return data_list

directory_path = 'content/depth'
depth_map_list = load_npy_files(directory_path)

# 로드된 데이터 확인
# for i, data in enumerate(depth_list):
#     print(f"Data {i+1}: {data}")
print(len(depth_map_list))


180


In [3]:
fx, fy, cx, cy = 800, 800, 640, 360
height, width = depth_map_list[0].shape

points3d_map_list = [[] for _ in range(len(depth_map_list))]
for i, depth in enumerate(depth_map_list):
    for v in range(height):
        for u in range(width):
            d = depth[v, u]
            if d == 0:
                continue
            Z = d
            X = (u - cx) * Z / fx
            Y = (v - cy) * Z / fy
            points3d_map_list[i].append((X, Y, Z))

    points3d_map_list[i] = np.array(points3d_map_list[i])


In [4]:
import cv2
cap = cv2.VideoCapture('content/test.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

In [5]:
print(fps)
print(height, width)

15.0
720 1280


영상 이미지 크기와 depth map shape이 달라서 tracked_data resizing 진행  
tracked_data는 이미지 크기 기준  

In [36]:
tracked_data = np.load('content/tracker/tracked_data.npy', allow_pickle=True)
tracking_len = len(tracked_data)
print(tracking_len)

print(height, width)

img = cv2.imread('content/depth/000000_depth.png')
img_height, img_width, _ = img.shape
print(img_height, img_width)

resizing = [height/img_height, width/img_width]
print(resizing)

150
720 1280
1080 1920
[0.6666666666666666, 0.6666666666666666]


In [41]:
def cal_center_points(vehicle):
    center_points = [[] for _ in range(len(tracked_data))]
    for i, data in enumerate(tracked_data):
        if vehicle in data:
            points = data[vehicle]
            v, u = (points[0]+points[2])*resizing[0]/2, (points[1]+points[3])*resizing[1]/2
            center = int(v), int(u)
            # index error 검사
            # if center[0] > width-1 or center[1] > height-1:
            #     print(f"{vehicle} {i}th frame out of range({height}, {width}): point is {points}, center is {center}")
            center_points[i].append(center)
        else:
            center = None, None
            center_points[i].append(center)
    return center_points

print(center_points[0]) - [(959, 495)]  
print(points3d_map_list[0].shape) - (921600, 3)  

In [79]:
# 특정 포인트의 이동 거리 구하기 (없는 프레임 건너뜀)
"""
velocity = []
for i, center in enumerate(center_points):
    if center[0][0] is None:
        continue
    else:
        v, u = center[0][0], center[0][1]
        point_idx = int(v * height + u)

        next = i+1
        while next <= len(center_points)-2 and center_points[next][0][0] is None:

            next+=1
        v, u = center_points[next][0][0], center_points[next][0][1]
        if v is None:
            continue
        next_idx = int(v * height + u)
        
        delx = np.linalg.norm(np.array(points3d_map_list[i][point_idx]) - np.array(points3d_map_list[next][next_idx]))
        velocity.append(delx*fps*(next-i))
        # x/(frame*#f) * frame/s * #f 
"""


'\nvelocity = []\nfor i, center in enumerate(center_points):\n    if center[0][0] is None:\n        continue\n    else:\n        v, u = center[0][0], center[0][1]\n        point_idx = int(v * height + u)\n\n        next = i+1\n        while next <= len(center_points)-2 and center_points[next][0][0] is None:\n\n            next+=1\n        v, u = center_points[next][0][0], center_points[next][0][1]\n        if v is None:\n            continue\n        next_idx = int(v * height + u)\n        \n        delx = np.linalg.norm(np.array(points_3d[i][point_idx]) - np.array(points_3d[next][next_idx]))\n        velocity.append(delx*fps*(next-i))\n        # x/(frame*#f) * frame/s * #f \n'

### 프레임마다 velocity 구하기  
구할 수 없는 경우에는 None 값  

In [30]:
# 특정 포인트의 이동 거리 구하기
def cal_velocity(center_points):
    velocity = []
    for i, center in enumerate(center_points):
        if center[0][0] is None:
            velocity.append(None)
            continue
        else:
            v, u = center[0][0], center[0][1]
            point_idx = v * height + u

            next = i
            while next <= tracking_len-2 and center_points[next][0][0] is None:
                next+=1
            
            try:
                v, u = center_points[next][0][0], center_points[next][0][1]
            except Exception as e:
                print(f"next is {next}: {e}")

            if v is None:
                velocity.append(None)
                continue
            next_idx = v * height + u

            try:
                delx = np.linalg.norm(np.array(points3d_map_list[i][point_idx]) - np.array(points3d_map_list[next+1][next_idx]))
                velocity.append(delx*fps*(next-i))

                # x/(frame*#f) * frame/s * #f
            except Exception as e:
                print(f"v is {v} u is {u}") 
                print(e)
    return velocity
    

In [40]:
all_vehicles = set(key for data in tracked_data for key in data.keys())
velocity_dic = {}
for vehicle in all_vehicles:
    center_points = cal_center_points(vehicle)
    velocity = cal_velocity(center_points)
    print(f"{vehicle} velocity: {velocity}")
    print(len(velocity))
    velocity_dic[vehicle] = velocity

vehicle8 velocity: [None, None, None, None, None, None, None, None, None, None, None, None, np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None,